# CodeGuardLLM: week-one GPU pilot
Select a GPU runtime. Upload `codeguard-week1.zip` built by `scripts/package_cloud.py`. In Colab, use the Files sidebar. In Kaggle, attach the archive as a dataset and set ARCHIVE below to its path.
This notebook generates and detects code only. Execute correctness tests later in the supplied Docker evaluator.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile
BASE = Path('/kaggle/working' if Path('/kaggle').exists() else '/content')
ARCHIVE = BASE / 'codeguard-week1.zip'  # Edit for a Kaggle input dataset
PROJECT = BASE / 'CodeGuardLLM'
if not PROJECT.exists():
    if not ARCHIVE.is_file():
        raise FileNotFoundError(f'Upload the project archive and set ARCHIVE: {ARCHIVE}')
    with zipfile.ZipFile(ARCHIVE) as archive:
        for item in archive.infolist():
            if not (BASE / item.filename).resolve().is_relative_to(PROJECT.resolve()):
                raise ValueError(f'Unexpected archive path: {item.filename}')
        archive.extractall(BASE)
assert (PROJECT / 'pyproject.toml').is_file(), 'Incorrect project archive'
os.chdir(PROJECT)
print(PROJECT)


Use an isolated Python environment to avoid mixing cloud preinstalled libraries with the experiment. Installation and first model download can take several minutes.

In [ ]:
ENV = PROJECT / '.venv'
if not (ENV / 'bin/python').exists():
    subprocess.run([sys.executable, '-m', 'venv', str(ENV)], check=True)
PYTHON = str(ENV / 'bin/python')
def run(*args):
    subprocess.run([PYTHON, *map(str, args)], cwd=PROJECT, check=True)
run('-m', 'pip', 'install', '-e', '.[ml,data]')
run('scripts/bootstrap_markllm.py')
run('-c', "import torch; print('GPU:', torch.cuda.get_device_name() if torch.cuda.is_available() else 'none'); assert torch.cuda.is_available(), 'Select a GPU runtime before continuing'")
run('-m', 'codeguard', 'validate', '--check-markllm')
run('-m', 'unittest', 'discover', '-s', 'tests', '-v')


## Choose persistent output
In Colab, this cell mounts your Google Drive after interactive authorization and saves each sample there. In Kaggle, output goes to `/kaggle/working`; download the archive below and save notebook outputs before ending the session. Kaggle working files are not a durable checkpoint across discarded sessions.
To resume a downloaded run, extract its `week1` directory into your chosen run parent. Never run two notebook sessions against the same output directory. A changed GPU or package environment requires a new run directory.

In [ ]:
if BASE == Path('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_DIR = Path('/content/drive/MyDrive/CodeGuardLLM/runs/week1')
else:
    RUN_DIR = BASE / 'runs/week1'
print('Output:', RUN_DIR)


In [ ]:
run('-m', 'codeguard', 'generate', '--config', 'configs/week1.yaml', '--output-dir', RUN_DIR)
run('-m', 'codeguard', 'summarize', '--run-dir', RUN_DIR)
summary = json.loads((RUN_DIR / 'summary.json').read_text())
assert summary['generated_samples'] == 20
print('Twenty completions saved. Pass@1 remains null until Docker evaluation.')


In [ ]:
# Also usable after an interrupted run to export the completed samples.
EXPORT = shutil.make_archive(str(BASE / 'codeguard-week1-results'), 'zip', RUN_DIR.parent, RUN_DIR.name)
print('Download:', EXPORT)
if BASE == Path('/content'):
    from google.colab import files
    files.download(EXPORT)


## Evaluate on your computer
Extract the results archive into the project's `runs` directory. Start Docker Desktop's Linux engine, then follow the README to build `codeguard-eval:week1`, run Docker smoke tests, evaluate, and summarize.
The ten development tasks test infrastructure; they are not HumanEval results or evidence of watermark reliability.